# Partitioned LDSC

Goal: run partitioned LDSC in the refactored codebase by building query annotations, computing baseline-plus-query LD scores from an R2-table reference panel, and regressing one trait on that partitioned design.

Path-token rules used below:
- use `@` for chromosome suites such as `baseline.@.annot.gz`
- use globs when the filenames do not follow the simple chromosome-suffix convention
- scalar inputs still resolve to exactly one file
- output prefixes remain literal destinations


In [ ]:
import os
import sys

# Set this explicitly for your environment.
SRC_DIR = "directory/to/src"

if not os.path.isdir(SRC_DIR):
    raise FileNotFoundError(f"Set SRC_DIR to an existing path before running the notebook: {SRC_DIR}")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

## Process Annotation Files

In [ ]:
from ldsc import GlobalConfig, get_global_config, run_bed_to_annot, set_global_config

In [ ]:
# Set these explicitly for your environment.
OUTPUT_DIR = "directory/for/outputs"

BED_FILES = "path/to/raw_bed/*.bed"  # use glob pattern to specify all bed files in the directory
BASELINE_ANNOT_PATHS = "directory/to/baseline_annot/baseline.@.annot.gz"
QUERY_ANNOT_OUTPUT_DIR = "directory/to/processed_query_annot"

set_global_config(GlobalConfig(snp_identifier="chr_pos", genome_build="hg19"))  # shared genome build of all input files
GLOBAL_CONFIG = get_global_config()

In [ ]:
run_bed_to_annot(
    bed_files=BED_FILES,
    baseline_annot_paths=BASELINE_ANNOT_PATHS,
    output_dir=QUERY_ANNOT_OUTPUT_DIR,
)

## Calculate LD Scores

In [ ]:
from ldsc import (
    AnnotationBuildConfig,
    AnnotationBuilder,
    AnnotationSourceSpec,
    GlobalConfig,
    LDScoreCalculator,
    LDScoreConfig,
    RefPanelLoader,
    RefPanelSpec,
    RegressionConfig,
    RegressionRunner,
    get_global_config,
    set_global_config,
)

In [ ]:
set_global_config(GlobalConfig(snp_identifier="chr_pos", genome_build="hg19"))
GLOBAL_CONFIG = get_global_config()

OUTPUT_DIR = "directory/for/outputs"
BASELINE_ANNOT_PATHS = "path/to/baseline_annot/baseline.@.annot.gz"
QUERY_ANNOT_PATHS = "path/to/query_annot/query.@.annot.gz"
R2_TABLE_PATHS = "path/to/ref_panel/parquet/ld/EUR_chr@_LD.parquet"
LD_WIND_CM = 1.0

In [ ]:
annotation_bundle = AnnotationBuilder(GLOBAL_CONFIG, AnnotationBuildConfig()).run(
    AnnotationSourceSpec(
        baseline_annot_paths=BASELINE_ANNOT_PATHS,
        query_annot_paths=QUERY_ANNOT_PATHS,
    )
)

ref_panel = RefPanelLoader(GLOBAL_CONFIG).load(
    RefPanelSpec(
        backend="parquet_r2",
        r2_table_paths=R2_TABLE_PATHS,
        chromosomes=tuple(annotation_bundle.chromosomes),
    )
)

ldscore_result = LDScoreCalculator().run(
    annotation_bundle=annotation_bundle,
    ref_panel=ref_panel,
    ldscore_config=LDScoreConfig(ld_wind_cm=LD_WIND_CM),
    global_config=GLOBAL_CONFIG,
)

runner = RegressionRunner(regression_config=RegressionConfig())

Since the annotation files and reference panel do not cover exactly the same SNP universe, the LD-score run can only use SNPs that exist in both:
- the annotation bundle
- the R2 table

In [ ]:
before = annotation_bundle.metadata["CHR"].value_counts().sort_index()
after = ldscore_result.reference_metadata["CHR"].value_counts().sort_index()

comparison = (
    before.rename("annotation_bundle")
    .to_frame()
    .join(after.rename("retained_in_ldscore"), how="outer")
    .fillna(0)
    .astype(int)
)
comparison["dropped"] = comparison["annotation_bundle"] - comparison["retained_in_ldscore"]
comparison

## Estimate Partitioned Heritability

In [ ]:
from ldsc import load_sumstats

In [ ]:
SUMSTATS_PATH = "path/to/normalized_sumstats/trait.sumstats.gz"
TRAIT_NAME = "trait"
PARTITIONED_OUTPUT_PATH = f"{OUTPUT_DIR}/pldsc_results/partitioned_h2.tsv"

In [ ]:
sumstats = load_sumstats(
    SUMSTATS_PATH,
    trait_name=TRAIT_NAME,
)

partitioned = runner.estimate_partitioned_h2_batch(
    sumstats,
    ldscore_result,
    annotation_bundle,
)

partitioned.to_csv(PARTITIONED_OUTPUT_PATH, sep="\t", index=False)
partitioned